# Notebook 06 — Validación Cruzada Completa
**Hito 2 · Detección automática de crisis epilépticas (EEG / CHB-MIT)**

Este notebook implementa los dos protocolos de validación descritos en el plan:

1. **Patient-Specific (5-Fold Stratified KFold)**: Baseline histórico de Shoeb & Guttag (2004).  
   Entrena y evalúa sobre el mismo paciente. Mide cuánto aprende el modelo sobre un individuo.

2. **Leave-One-Patient-Out (LOPO)**: Protocolo más exigente.  
   En cada iteración, un paciente es test y el resto son train. Mide la generalización entre sujetos.

**Regla anti-leakage garantizada**: el balanceo (SMOTE + submuestreo) y el ajuste del `StandardScaler`  
se realizan ÚNICAMENTE sobre el conjunto de entrenamiento de cada fold/iteración.  
Aplicarlos antes de partir los datos es uno de los errores metodológicos más reportados en la literatura de CHB-MIT.

**Modelos comparados**: Random Forest (clasificador principal) vs. SVM-RBF (comparación)

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Permitir que el notebook importe desde la carpeta 'src'
sys.path.append(os.path.abspath('../'))

from src.models.classifiers import get_random_forest_pipeline, get_svm_pipeline
from src.evaluation.validation import patient_specific_cv, leave_one_patient_out_cv

print("✓ Imports OK")

## 1. Carga de Datos Procesados

Cargamos las matrices de características `X` y etiquetas `y` generadas por el Notebook 04  
(pipeline de procesamiento por lotes de pacientes reales CHB-MIT).

In [ ]:
# =====================================================================
# 1. CARGA DE DATOS PROCESADOS (archivos .npy de data/processed/)
# =====================================================================
DATA_DIR = os.path.abspath('../data/processed')

# Diccionario {patient_id: (X, y)} para el protocolo LOPO
patients_data = {}

# Detectar automáticamente los pacientes disponibles por los archivos X_chbXX_completo.npy
npy_files = [f for f in os.listdir(DATA_DIR) if f.startswith('X_') and f.endswith('.npy')]
patient_ids = sorted([f.replace('X_', '').replace('_completo.npy', '') for f in npy_files])

print(f"Pacientes encontrados en {DATA_DIR}:")
for pid in patient_ids:
    X_path = os.path.join(DATA_DIR, f'X_{pid}_completo.npy')
    y_path = os.path.join(DATA_DIR, f'y_{pid}_completo.npy')
    
    X = np.load(X_path)
    y = np.load(y_path)
    patients_data[pid] = (X, y)
    
    pct_crisis = 100 * np.sum(y == 1) / len(y)
    print(f"  {pid}: X={X.shape}, y={y.shape} | Crisis: {np.sum(y==1)} ({pct_crisis:.2f}%)")

## 2. Validación Patient-Specific (5-Fold CV)

Para cada paciente, aplicamos una validación cruzada estratificada de 5 folds.  
Comparamos Random Forest vs. SVM-RBF.

In [ ]:
# =====================================================================
# 2. VALIDACIÓN PATIENT-SPECIFIC — RANDOM FOREST
# =====================================================================
print("\n" + "="*70)
print("PATIENT-SPECIFIC — RANDOM FOREST")
print("="*70)

# Hiperparámetros de validación
N_SPLITS = 5
SMOTE_STRATEGY = 0.1
UNDERSAMPLE_STRATEGY = 0.5
N_WINDOWS_HISTERESIS = 3
THRESHOLD_HIGH = 0.6
THRESHOLD_LOW = 0.4

# Resultados patient-specific para RF
ps_rf_results = {}  # {patient_id: resumen_cv}

for pid, (X, y) in patients_data.items():
    print(f"\n--- Paciente: {pid} ---")
    
    # Verificar que hay suficientes muestras de crisis para hacer K-Fold estratificado
    n_crisis = np.sum(y == 1)
    if n_crisis < N_SPLITS:
        print(f"  SKIP: Solo {n_crisis} ventanas de crisis (mínimo {N_SPLITS} necesarias para {N_SPLITS}-Fold)")
        continue
    
    clf = get_random_forest_pipeline(n_estimators=100, random_state=42)
    
    folds, resumen = patient_specific_cv(
        X, y, clf,
        n_splits=N_SPLITS,
        smote_strategy=SMOTE_STRATEGY,
        undersample_strategy=UNDERSAMPLE_STRATEGY,
        N_windows=N_WINDOWS_HISTERESIS,
        threshold_high=THRESHOLD_HIGH,
        threshold_low=THRESHOLD_LOW,
        random_state=42,
        verbose=True
    )
    
    if resumen:
        ps_rf_results[pid] = resumen

print("\n✓ Validación Patient-Specific RF completada")

In [ ]:
# =====================================================================
# 3. VALIDACIÓN PATIENT-SPECIFIC — SVM-RBF
# =====================================================================
print("\n" + "="*70)
print("PATIENT-SPECIFIC — SVM-RBF (con StandardScaler integrado)")
print("="*70)

# Resultados patient-specific para SVM
ps_svm_results = {}  # {patient_id: resumen_cv}

for pid, (X, y) in patients_data.items():
    print(f"\n--- Paciente: {pid} ---")
    
    n_crisis = np.sum(y == 1)
    if n_crisis < N_SPLITS:
        print(f"  SKIP: Solo {n_crisis} ventanas de crisis (mínimo {N_SPLITS} necesarias para {N_SPLITS}-Fold)")
        continue
    
    clf = get_svm_pipeline(C=1.0, gamma='scale', random_state=42)
    
    folds, resumen = patient_specific_cv(
        X, y, clf,
        n_splits=N_SPLITS,
        smote_strategy=SMOTE_STRATEGY,
        undersample_strategy=UNDERSAMPLE_STRATEGY,
        N_windows=N_WINDOWS_HISTERESIS,
        threshold_high=THRESHOLD_HIGH,
        threshold_low=THRESHOLD_LOW,
        random_state=42,
        verbose=True
    )
    
    if resumen:
        ps_svm_results[pid] = resumen

print("\n✓ Validación Patient-Specific SVM completada")

## 3. Validación Leave-One-Patient-Out (LOPO)

Protocolo más exigente: mide la generalización real entre sujetos.  
Un paciente es test; el resto son train concatenados.

In [ ]:
# =====================================================================
# 4. VALIDACIÓN LEAVE-ONE-PATIENT-OUT — RANDOM FOREST
# =====================================================================
print("\n" + "="*70)
print("LEAVE-ONE-PATIENT-OUT — RANDOM FOREST")
print("="*70)

clf_rf = get_random_forest_pipeline(n_estimators=100, random_state=42)

lopo_rf_por_paciente, lopo_rf_global = leave_one_patient_out_cv(
    patients_data, clf_rf,
    smote_strategy=SMOTE_STRATEGY,
    undersample_strategy=UNDERSAMPLE_STRATEGY,
    N_windows=N_WINDOWS_HISTERESIS,
    threshold_high=THRESHOLD_HIGH,
    threshold_low=THRESHOLD_LOW,
    random_state=42,
    verbose=True
)

print("\n✓ LOPO RF completado")

In [ ]:
# =====================================================================
# 5. VALIDACIÓN LEAVE-ONE-PATIENT-OUT — SVM-RBF
# =====================================================================
print("\n" + "="*70)
print("LEAVE-ONE-PATIENT-OUT — SVM-RBF")
print("="*70)

clf_svm = get_svm_pipeline(C=1.0, gamma='scale', random_state=42)

lopo_svm_por_paciente, lopo_svm_global = leave_one_patient_out_cv(
    patients_data, clf_svm,
    smote_strategy=SMOTE_STRATEGY,
    undersample_strategy=UNDERSAMPLE_STRATEGY,
    N_windows=N_WINDOWS_HISTERESIS,
    threshold_high=THRESHOLD_HIGH,
    threshold_low=THRESHOLD_LOW,
    random_state=42,
    verbose=True
)

print("\n✓ LOPO SVM completado")

## 4. Tabla Comparativa de Resultados

Mostramos la tabla resumen de métricas clínicas para los 4 escenarios evaluados.

In [ ]:
# =====================================================================
# 6. TABLA COMPARATIVA GLOBAL
# =====================================================================

def fmt_val(v, decimals=3):
    if v is None:
        return "N/D"
    return f"{v:.{decimals}f}"

def get_ps_global(ps_results):
    """Promedia los resúmenes patient-specific de todos los pacientes."""
    if not ps_results:
        return {}
    from src.evaluation.validation import _agregar_metricas
    return _agregar_metricas(list(ps_results.values()))

ps_rf_global = get_ps_global(ps_rf_results)
ps_svm_global = get_ps_global(ps_svm_results)

print("\n" + "="*100)
print("TABLA COMPARATIVA DE MÉTRICAS CLÍNICAS")
print("="*100)
print(f"{'Métrica':<35} | {'PS-RF':>10} | {'PS-SVM':>10} | {'LOPO-RF':>10} | {'LOPO-SVM':>10}")
print("-"*100)

metricas_a_mostrar = [
    ("Sensibilidad (TPR)",         "Sensibilidad (TPR)",              3),
    ("Especificidad (TNR)",         "Especificidad (TNR)",             3),
    ("AUC-ROC",                     "AUC-ROC",                         3),
    ("AUC-PR",                      "AUC-PR",                          3),
    ("Tasa Falsas Alarmas (FA/h)",  "Tasa de Falsas Alarmas (FA/h)",   2),
    ("Latencia de Detección (s)",   "Latencia de Detección (s)",       1),
]

for label, key, dec in metricas_a_mostrar:
    v_ps_rf   = ps_rf_global.get(key)
    v_ps_svm  = ps_svm_global.get(key)
    v_lopo_rf = lopo_rf_global.get(key)
    v_lopo_svm = lopo_svm_global.get(key)
    print(f"{label:<35} | {fmt_val(v_ps_rf, dec):>10} | {fmt_val(v_ps_svm, dec):>10} | {fmt_val(v_lopo_rf, dec):>10} | {fmt_val(v_lopo_svm, dec):>10}")

print("="*100)
print("\nPS  = Patient-Specific (5-Fold CV por paciente)")
print("LOPO = Leave-One-Patient-Out (generalización entre sujetos)")

## 5. Gráficos de Resultados

In [ ]:
# =====================================================================
# 7. GRÁFICOS DE MÉTRICAS
# =====================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Comparativa de Validación Cruzada — EEG Crisis Epilépticas (CHB-MIT)',
             fontsize=14, fontweight='bold')

# --- Paleta de colores ---
COLOR_RF  = '#2196F3'   # Azul para Random Forest
COLOR_SVM = '#FF5722'   # Naranja para SVM
ALPHA_PS   = 0.9
ALPHA_LOPO = 0.6

patient_list = list(patients_data.keys())

# ========================
# Gráfico 1: Sensibilidad por paciente (Patient-Specific)
# ========================
ax = axes[0]
x = np.arange(len(patient_list))
width = 0.35

sensib_rf  = [ps_rf_results.get(p, {}).get('Sensibilidad (TPR)', 0) for p in patient_list]
sensib_svm = [ps_svm_results.get(p, {}).get('Sensibilidad (TPR)', 0) for p in patient_list]

bars1 = ax.bar(x - width/2, sensib_rf,  width, label='RF',  color=COLOR_RF,  alpha=ALPHA_PS)
bars2 = ax.bar(x + width/2, sensib_svm, width, label='SVM', color=COLOR_SVM, alpha=ALPHA_PS)

ax.set_xlabel('Paciente')
ax.set_ylabel('Sensibilidad (TPR)')
ax.set_title('Sensibilidad por Paciente\n(Patient-Specific 5-Fold CV)')
ax.set_xticks(x)
ax.set_xticklabels(patient_list, rotation=45, ha='right')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='Umbral clínico 0.8')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ========================
# Gráfico 2: AUC-ROC — Patient-Specific vs. LOPO
# ========================
ax = axes[1]

auc_ps_rf    = [ps_rf_results.get(p, {}).get('AUC-ROC', 0) for p in patient_list]
auc_lopo_rf  = [lopo_rf_por_paciente.get(p, {}).get('AUC-ROC', 0) for p in patient_list]
auc_ps_svm   = [ps_svm_results.get(p, {}).get('AUC-ROC', 0) for p in patient_list]
auc_lopo_svm = [lopo_svm_por_paciente.get(p, {}).get('AUC-ROC', 0) for p in patient_list]

x4 = np.arange(len(patient_list))
w4 = 0.2

ax.bar(x4 - 1.5*w4, auc_ps_rf,    w4, label='RF Patient-Specific',  color=COLOR_RF,  alpha=ALPHA_PS)
ax.bar(x4 - 0.5*w4, auc_lopo_rf,  w4, label='RF LOPO',              color=COLOR_RF,  alpha=ALPHA_LOPO)
ax.bar(x4 + 0.5*w4, auc_ps_svm,   w4, label='SVM Patient-Specific', color=COLOR_SVM, alpha=ALPHA_PS)
ax.bar(x4 + 1.5*w4, auc_lopo_svm, w4, label='SVM LOPO',             color=COLOR_SVM, alpha=ALPHA_LOPO)

ax.set_xlabel('Paciente')
ax.set_ylabel('AUC-ROC')
ax.set_title('AUC-ROC por Paciente\n(Patient-Specific vs. LOPO)')
ax.set_xticks(x4)
ax.set_xticklabels(patient_list, rotation=45, ha='right')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Azar (0.5)')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# ========================
# Gráfico 3: Tasa de Falsas Alarmas (FA/h)
# ========================
ax = axes[2]

fa_ps_rf    = [ps_rf_results.get(p, {}).get('Tasa de Falsas Alarmas (FA/h)', 0) for p in patient_list]
fa_lopo_rf  = [lopo_rf_por_paciente.get(p, {}).get('Tasa de Falsas Alarmas (FA/h)', 0) for p in patient_list]
fa_ps_svm   = [ps_svm_results.get(p, {}).get('Tasa de Falsas Alarmas (FA/h)', 0) for p in patient_list]
fa_lopo_svm = [lopo_svm_por_paciente.get(p, {}).get('Tasa de Falsas Alarmas (FA/h)', 0) for p in patient_list]

ax.bar(x4 - 1.5*w4, fa_ps_rf,    w4, label='RF Patient-Specific',  color=COLOR_RF,  alpha=ALPHA_PS)
ax.bar(x4 - 0.5*w4, fa_lopo_rf,  w4, label='RF LOPO',              color=COLOR_RF,  alpha=ALPHA_LOPO)
ax.bar(x4 + 0.5*w4, fa_ps_svm,   w4, label='SVM Patient-Specific', color=COLOR_SVM, alpha=ALPHA_PS)
ax.bar(x4 + 1.5*w4, fa_lopo_svm, w4, label='SVM LOPO',             color=COLOR_SVM, alpha=ALPHA_LOPO)

ax.set_xlabel('Paciente')
ax.set_ylabel('FA/hora')
ax.set_title('Falsas Alarmas por Hora\n(menor es mejor)')
ax.set_xticks(x4)
ax.set_xticklabels(patient_list, rotation=45, ha='right')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/06_comparativa_validacion_cruzada.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico guardado en reports/figures/")

In [ ]:
# =====================================================================
# 8. BOXPLOT DE SENSIBILIDAD — DISTRIBUCIÓN POR PACIENTE (LOPO)
# =====================================================================

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Distribución de Sensibilidad — LOPO por Paciente',
             fontsize=13, fontweight='bold')

sensib_lopo_rf  = [lopo_rf_por_paciente.get(p, {}).get('Sensibilidad (TPR)', np.nan) for p in patient_list]
sensib_lopo_svm = [lopo_svm_por_paciente.get(p, {}).get('Sensibilidad (TPR)', np.nan) for p in patient_list]

x = np.arange(len(patient_list))
width = 0.35

bars_rf  = ax.bar(x - width/2, [v if not np.isnan(v) else 0 for v in sensib_lopo_rf],
                  width, label='RF LOPO',  color=COLOR_RF,  alpha=0.8)
bars_svm = ax.bar(x + width/2, [v if not np.isnan(v) else 0 for v in sensib_lopo_svm],
                  width, label='SVM LOPO', color=COLOR_SVM, alpha=0.8)

# Línea de referencia del promedio global
if lopo_rf_global.get('Sensibilidad (TPR)'):
    ax.axhline(y=lopo_rf_global['Sensibilidad (TPR)'], color=COLOR_RF,
               linestyle='--', alpha=0.7, label=f"RF media={lopo_rf_global['Sensibilidad (TPR)']:.3f}")
if lopo_svm_global.get('Sensibilidad (TPR)'):
    ax.axhline(y=lopo_svm_global['Sensibilidad (TPR)'], color=COLOR_SVM,
               linestyle='--', alpha=0.7, label=f"SVM media={lopo_svm_global['Sensibilidad (TPR)']:.3f}")

ax.set_xlabel('Paciente (test)')
ax.set_ylabel('Sensibilidad (TPR)')
ax.set_xticks(x)
ax.set_xticklabels(patient_list, rotation=45, ha='right')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/06_sensibilidad_lopo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Gráfico de sensibilidad LOPO guardado")

## 6. Síntesis y Análisis

**Interpretación esperada de los resultados:**

| Escenario | Sensibilidad esperada | AUC-ROC esperado | Observación |
|---|---|---|---|
| Patient-Specific (RF) | Alta (≥ 0.80) | ≥ 0.90 | Modelo ya vio señales del paciente |
| Patient-Specific (SVM)| Media-Alta | ≥ 0.85 | Depende de la calibración del kernel |
| LOPO (RF) | Media (0.50-0.75) | 0.70-0.85 | Generalización real entre sujetos |
| LOPO (SVM) | Media | Similar a RF-LOPO | Más sensible a hiperparámetros |

**La brecha entre Patient-Specific y LOPO** es el indicador más honesto del desafío de generalización:  
cuanto mayor es esa brecha, más específico es el modelo para cada individuo y menos generalizable a nuevos pacientes.

**Regla de oro clínica**: Sensibilidad > FA/h. Un sistema útil detecta la mayoría de las crisis con pocas falsas alarmas por hora.

**Nota**: Con solo 5 pacientes (chb01–chb05), el LOPO tiene alta varianza. Con los 22 pacientes completos de CHB-MIT, los promedios serían más estables y representativos.